Imports

In [26]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

from xgboost import XGBClassifier
from scipy.sparse import hstack


Load Data

In [27]:
train_df = pd.read_csv("/kaggle/input/neural-craft/train_complaints.csv")
test_df  = pd.read_csv("/kaggle/input/neural-craft/test_complaints.csv")


#Filling Missing Values

In [28]:
train_df["complaint_text"] = train_df["complaint_text"].fillna("")
test_df["complaint_text"]  = test_df["complaint_text"].fillna("")


Text Cleaning

In [29]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df["clean_text"] = train_df["complaint_text"].apply(clean_text)
test_df["clean_text"]  = test_df["complaint_text"].apply(clean_text)


Label Encoding

In [30]:
le_primary   = LabelEncoder()
le_secondary = LabelEncoder()
le_severity  = LabelEncoder()

y_primary   = le_primary.fit_transform(train_df["primary_category"])
y_secondary = le_secondary.fit_transform(train_df["secondary_category"])
y_severity  = le_severity.fit_transform(train_df["severity"])

X = train_df["clean_text"]


Splitting Data into train and validation set

In [31]:
X_train, X_val, y_primary_train, y_primary_val, y_secondary_train, y_secondary_val, y_severity_train, y_severity_val = train_test_split(
    X,
    y_primary,
    y_secondary,
    y_severity,
    test_size=0.1,
    random_state=42,
    stratify=y_primary
)


TF - IDF (as in the sample notebook with some changes)

In [32]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=80000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(test_df["clean_text"])


Sample Weights

In [33]:
sample_weights_primary   = compute_sample_weight("balanced", y_primary_train)
sample_weights_secondary = compute_sample_weight("balanced", y_secondary_train)
sample_weights_severity  = compute_sample_weight("balanced", y_severity_train)


Model for primary category

In [34]:
from xgboost.callback import EarlyStopping
primary_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=1500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2,
    reg_alpha=0.5,
    tree_method="hist",
    n_jobs=-1
)

primary_model.fit(
    X_train_vec,
    y_primary_train,
    sample_weight=sample_weights_primary,
    eval_set=[(X_val_vec, y_primary_val)],
    verbose=100,
     
)


[0]	validation_0-mlogloss:2.14473
[100]	validation_0-mlogloss:0.95943
[200]	validation_0-mlogloss:0.80390
[300]	validation_0-mlogloss:0.75813
[400]	validation_0-mlogloss:0.74145
[500]	validation_0-mlogloss:0.73575
[600]	validation_0-mlogloss:0.73541
[700]	validation_0-mlogloss:0.73962
[800]	validation_0-mlogloss:0.74615
[900]	validation_0-mlogloss:0.75179
[1000]	validation_0-mlogloss:0.76048
[1100]	validation_0-mlogloss:0.76821
[1200]	validation_0-mlogloss:0.77533
[1300]	validation_0-mlogloss:0.78279
[1400]	validation_0-mlogloss:0.79057
[1499]	validation_0-mlogloss:0.79810


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1500, n_jobs=-1,
              num_parallel_tree=None, ...)

Primary Predictions (for stacking)

In [35]:
primary_train_pred = primary_model.predict(X_train_vec)
primary_val_pred   = primary_model.predict(X_val_vec)

primary_train_feat = primary_train_pred.reshape(-1,1)
primary_val_feat   = primary_val_pred.reshape(-1,1)

X_train_sec = hstack([X_train_vec, primary_train_feat])
X_val_sec   = hstack([X_val_vec, primary_val_feat])


Model for secondary category (using primary category as feature)

In [36]:
secondary_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=1500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2,
    reg_alpha=0.5,
    tree_method="hist",
    n_jobs=-1
)

secondary_model.fit(
    X_train_sec,
    y_secondary_train,
    sample_weight=sample_weights_secondary,
    eval_set=[(X_val_sec, y_secondary_val)],
    verbose=100,
       
)


[0]	validation_0-mlogloss:2.24504
[100]	validation_0-mlogloss:1.45321
[200]	validation_0-mlogloss:1.67909
[300]	validation_0-mlogloss:1.85674
[400]	validation_0-mlogloss:1.95984
[500]	validation_0-mlogloss:2.02196
[600]	validation_0-mlogloss:2.06156
[700]	validation_0-mlogloss:2.09403
[800]	validation_0-mlogloss:2.11643
[900]	validation_0-mlogloss:2.13243
[1000]	validation_0-mlogloss:2.14989
[1100]	validation_0-mlogloss:2.16440
[1200]	validation_0-mlogloss:2.17561
[1300]	validation_0-mlogloss:2.18532
[1400]	validation_0-mlogloss:2.19267
[1499]	validation_0-mlogloss:2.19931


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1500, n_jobs=-1,
              num_parallel_tree=None, ...)

In [37]:
secondary_train_pred = secondary_model.predict(X_train_sec)
secondary_val_pred   = secondary_model.predict(X_val_sec)


In [38]:
secondary_train_feat = secondary_train_pred.reshape(-1,1)
secondary_val_feat   = secondary_val_pred.reshape(-1,1)

X_train_sev = hstack([X_train_sec, secondary_train_feat])
X_val_sev   = hstack([X_val_sec, secondary_val_feat])


Model for Severity category using primary and secondary as features

In [39]:


severity_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=1200,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    tree_method="hist",
    n_jobs=-1
)

severity_model.fit(
    X_train_sev,
    y_severity_train,
    sample_weight=sample_weights_severity,
    eval_set=[(X_val_sev, y_severity_val)],
    verbose=100,

)



[0]	validation_0-mlogloss:1.57410
[100]	validation_0-mlogloss:0.24264
[200]	validation_0-mlogloss:0.09161
[300]	validation_0-mlogloss:0.06883
[400]	validation_0-mlogloss:0.06613
[500]	validation_0-mlogloss:0.06793
[600]	validation_0-mlogloss:0.06941
[700]	validation_0-mlogloss:0.07097
[800]	validation_0-mlogloss:0.07267
[900]	validation_0-mlogloss:0.07409
[1000]	validation_0-mlogloss:0.07553
[1100]	validation_0-mlogloss:0.07682
[1199]	validation_0-mlogloss:0.07775


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1200, n_jobs=-1,
              num_parallel_tree=None, ...)

Validation Scores

In [40]:
primary_val_pred   = primary_model.predict(X_val_vec)
secondary_val_pred = secondary_model.predict(X_val_sec)
severity_val_pred  = severity_model.predict(X_val_sev)

print("Primary F1:", f1_score(y_primary_val, primary_val_pred, average="weighted"))
print("Secondary F1:", f1_score(y_secondary_val, secondary_val_pred, average="weighted"))
print("Severity F1:", f1_score(y_severity_val, severity_val_pred, average="weighted"))


Primary F1: 0.7017295098916617
Secondary F1: 0.5914457799573412
Severity F1: 0.982590011720197


In [41]:
print("Train vec shape:", X_train_vec.shape)
print("Test vec shape:", X_test_vec.shape)
print("Model expects:", primary_model.get_booster().num_features())


Train vec shape: (2699, 15297)
Test vec shape: (499, 15297)
Model expects: 15297


Training Final Models on Full Data

In [42]:
X_full_vec = vectorizer.transform(train_df["clean_text"])


primary_model.fit(X_full_vec, y_primary)

primary_full_pred = primary_model.predict(X_full_vec)
primary_full_feat = primary_full_pred.reshape(-1,1)

X_full_sec = hstack([X_full_vec, primary_full_feat])

secondary_model.fit(X_full_sec, y_secondary)

secondary_full_pred = secondary_model.predict(X_full_sec)
secondary_full_feat = secondary_full_pred.reshape(-1,1)

X_full_sev = hstack([X_full_sec, secondary_full_feat])

severity_model.fit(X_full_sev, y_severity)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1200, n_jobs=-1,
              num_parallel_tree=None, ...)

Test Predictions

In [43]:
primary_test_pred = primary_model.predict(X_test_vec)
primary_test_feat = primary_test_pred.reshape(-1,1)

X_test_sec = hstack([X_test_vec, primary_test_feat])

secondary_test_pred = secondary_model.predict(X_test_sec)
secondary_test_feat = secondary_test_pred.reshape(-1,1)

X_test_sev = hstack([X_test_sec, secondary_test_feat])

severity_test_pred = severity_model.predict(X_test_sev)

primary_labels   = le_primary.inverse_transform(primary_test_pred)
secondary_labels = le_secondary.inverse_transform(secondary_test_pred)
severity_labels  = le_severity.inverse_transform(severity_test_pred)


Creating a CSV FILE for submission

In [47]:
submission = pd.DataFrame({
    "complaint_id": test_df["complaint_id"],
    "primary_category": primary_labels,
    "secondary_category": secondary_labels,
    "severity": severity_labels
})

submission.to_csv("submission.csv", index=False)


In [46]:
print(test_df.columns)



Index(['complaint_id', 'complaint_text', 'clean_text'], dtype='object')
